In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings(action='ignore')

# ==============================================================================
# 1. Read data & Drop IDs
# ==============================================================================
test_type = 'external'

train = pd.read_csv('/media/data3/users/tpp/MIMICIV_AKI_AMI/Thai/MIMIC data/AKI_patients_with_features_train.csv')
if test_type == 'internal_MIMIC':
    test = pd.read_csv('/media/data3/users/tpp/MIMICIV_AKI_AMI/Thai/MIMIC data/AKI_patients_with_features_test.csv')
elif test_type == 'external':
    test = pd.read_csv('AKI_patients_with_features_eICU.csv')

# Drop index và id
for df in [train, test]:
    if 'Unnamed: 0' in df.columns:
        df.drop('Unnamed: 0', axis=1, inplace=True)
    id_cols = ['subject_id', 'hadm_id', 'stay_id', 'patientunitstayid']
    existing_id_cols = [col for col in id_cols if col in df.columns]
    df.drop(existing_id_cols, axis=1, inplace=True)

missing_in_test = set(train.columns) - set(test.columns)
for c in missing_in_test:
    test[c] = np.nan

test = test[train.columns]

# ==============================================================================
# 2. Xóa các cột có > 50% missing
# ==============================================================================
removed_cols = [col for col in train.columns if train[col].isnull().mean() > 0.5]
train.drop(removed_cols, axis=1, inplace=True)
test.drop(removed_cols, axis=1, inplace=True, errors='ignore')
print(f"Removed {len(removed_cols)} columns with >50% missing: {removed_cols}")

# ==============================================================================
# 3. Loại các bệnh nhân thiếu quá nhiều dữ liệu (>20%)
# ==============================================================================
# n_cols = len(train.columns) - 1 
# threshold = 0.8
# min_non_null = int(threshold * n_cols)
# train_before = len(train)
# test_before = len(test)

# train = train.dropna(thresh=min_non_null).reset_index(drop=True)
# test = test.dropna(thresh=min_non_null).reset_index(drop=True)

# print(f"Row removal: train {train_before} → {len(train)}, test {test_before} → {len(test)}")

# ==============================================================================
# 4. Encode categorical features
# ==============================================================================
drop_cat_cols = ['first_careunit', 'race']
for df in [train, test]:
    existing_drop = [c for c in drop_cat_cols if c in df.columns]
    df.drop(existing_drop, axis=1, inplace=True)
    
    if 'gender' in df.columns:
        df['gender'] = df['gender'].str.upper().map({'F': 0, 'M': 1})
    
    if 'first_hosp_stay' in df.columns:
        df['first_hosp_stay'] = pd.to_numeric(df['first_hosp_stay'], errors='coerce')

Removed 16 columns with >50% missing: ['albumin_median', 'nlr_max', 'bands_max', 'ntprobnp_max', 'fibrinogen_median', 'ck_cpk_max', 'ld_ldh_max', 'alt_max', 'ast_max', 'bilirubin_total_max', 'alp_median', 'amylase_median', 'peep_max', 'fio2_max', 'plateau_pressure_max', 'tidal_volume_observed_max']


In [8]:
# ==============================================================================
# 5. External Validation, Standardization, and KNN Imputation
# ==============================================================================
OUTPUT_DIR = "AKI_dataset/"
OUTPUT_DIR_TREE = "AKI_dataset_for_tree_model/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR_TREE, exist_ok=True)

y_train = train['aki_label']
X_train = train.drop(['aki_label'], axis=1)

y_test = test['aki_label']
X_test = test.drop(['aki_label'], axis=1)

print(f"\nX_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

categorical_cols = [
    'gender', 'first_hosp_stay', 'use_aspirin', 'has_acei', 'use_vancomycin_iv',
    'has_mcs_device', 'has_pa_catheter', 'is_femoral_access',
    'is_afib', 'is_paced', 'is_high_grade_block', 'invasive_vent_flag',
    'non_invasive_vent_flag', 'vaso_flag', 'infection_suspected_flag',
    'blood_culture_flag', 'gcs_unable_max',
]

num_cols = [
    'admission_age', 'cnt_systemic_antibiotics', 'cnt_invasive_lines',
    'weight', 'height', 'heart_rate_max', 'sys_bp_min', 'mean_bp_min',
    'dias_bp_min', 'resp_rate_max', 'temperature_max', 'glucose_vitalsign_max',
    'creatinine_min', 'bun_max', 'bicarbonate_min', 'potassium_chemistry_max',
    'aniongap_max', 'glucose_chemistry_max', 'urineoutput_12h_sum', 'hemoglobin_min',
    'hematocrit_min', 'wbc_max', 'platelet_median', 'rdw_median', 'troponin_t_max',
    'ck_mb_max', 'inr_median', 'pt_median', 'ptt_median', 'gcs_sin', 'gcs_motor_min',
    'norepinephrine_equivalent_dose_max', 'fio2_max',
    'albumin_median', 'nlr_max', 'bands_max', 'ntprobnp_max',
    'ck_cpk_max', 'ld_ldh_max', 'alt_max', 'ast_max', 'bilirubin_total_max',
    'alp_median', 'amylase_median'
]

categorical_cols = [col for col in categorical_cols if col in X_train.columns]
num_cols = [col for col in num_cols if col in X_train.columns]

# Check coverage
all_defined_cols = set(categorical_cols + num_cols)
all_X_cols = set(X_train.columns)
missing_in_definition = all_X_cols - all_defined_cols
extra_in_definition = all_defined_cols - all_X_cols

print(f"\ncategorical_cols: {len(categorical_cols)}")
print(f"num_cols: {len(num_cols)}")
if missing_in_definition:
    print(f"⚠️  Columns NOT defined: {missing_in_definition}")
if extra_in_definition:
    print(f"⚠️  Extra columns defined: {extra_in_definition}")

# Define capping columns
cap_99_only_cols = [
    'urineoutput_12h_sum', 'norepinephrine_equivalent_dose_max', 
    'peep_max', 'plateau_pressure_max', 'tidal_volume_observed_max',
    'cnt_systemic_antibiotics', 'cnt_invasive_lines'
]

winsorize_cols = [
    'admission_age', 'weight', 'height',
    'heart_rate_max', 'sys_bp_min', 'mean_bp_min', 'dias_bp_min', 
    'resp_rate_max', 'temperature_max', 'glucose_vitalsign_max',
    'creatinine_min', 'bun_max', 'bicarbonate_min', 'potassium_chemistry_max', 
    'aniongap_max', 'albumin_median', 'glucose_chemistry_max', 
    'hemoglobin_min', 'hematocrit_min', 'wbc_max', 'platelet_median', 
    'rdw_median', 'nlr_max', 'bands_max', 'troponin_t_max', 'ck_mb_max', 
    'ntprobnp_max', 'inr_median', 'pt_median', 'ptt_median', 'fibrinogen_median', 
    'ck_cpk_max', 'ld_ldh_max', 'alt_max', 'ast_max', 'bilirubin_total_max', 
    'alp_median', 'amylase_median'
]

print(f'\n{"="*60}')
print("EXTERNAL VALIDATION SETUP: TRAIN (MIMIC) → TEST (eICU)")
print(f'{"="*60}')

print(f"  Train label: AKI=1 {y_train.mean()*100:.1f}%, AKI=0 {(1-y_train.mean())*100:.1f}%")
print(f"  Test label:  AKI=1 {y_test.mean()*100:.1f}%, AKI=0 {(1-y_test.mean())*100:.1f}%")

# Cap outliers
print("\n--- Capping outliers ---")
for col in cap_99_only_cols:
    if col in X_train.columns:
        p99 = X_train[col].quantile(0.99)
        X_train[col] = np.where(X_train[col] > p99, p99, X_train[col])
        if col in X_test.columns:
            X_test[col] = np.where(X_test[col] > p99, p99, X_test[col])

for col in winsorize_cols:
    if col in X_train.columns:
        p01 = X_train[col].quantile(0.01)
        p99 = X_train[col].quantile(0.99)
        X_train[col] = X_train[col].clip(lower=p01, upper=p99)
        if col in X_test.columns:
            X_test[col] = X_test[col].clip(lower=p01, upper=p99)

print("✓ Outlier capping completed")

# Save tree model data
file_train_tree = os.path.join(OUTPUT_DIR_TREE, "train.csv")
file_test_tree = os.path.join(OUTPUT_DIR_TREE, "test.csv")
pd.concat([X_train, y_train], axis=1).to_csv(file_train_tree, index=False)
pd.concat([X_test, y_test], axis=1).to_csv(file_test_tree, index=False)
print(f"\n[Tree Model] Saved: {file_train_tree}, {file_test_tree}")

print(f"\nBefore categorical imputation:")
print(f"  Train missing: {X_train[categorical_cols].isnull().sum().sum()}")
print(f"  Test missing:  {X_test[categorical_cols].isnull().sum().sum()}")

cat_cols_fill_minus1 = ['gender', 'first_hosp_stay', 'gcs_unable_max']
cat_cols_fill_minus1 = [col for col in cat_cols_fill_minus1 if col in X_train.columns]

for df in [X_train, X_test]:
    df[cat_cols_fill_minus1] = df[cat_cols_fill_minus1].fillna(-1)

cat_cols_fill_0 = [col for col in categorical_cols if col not in cat_cols_fill_minus1]
for df in [X_train, X_test]:
    df[cat_cols_fill_0] = df[cat_cols_fill_0].fillna(0)

print(f"After categorical imputation:")
print(f"  Filled {len(cat_cols_fill_minus1)} cols with -1 (Unknown)")
print(f"  Filled {len(cat_cols_fill_0)} cols with 0 (Absence)")

# ✓ VERIFY no remaining categorical NaN
assert X_train[categorical_cols].isnull().sum().sum() == 0, "Train still has categorical NaN!"
assert X_test[categorical_cols].isnull().sum().sum() == 0, "Test still has categorical NaN!"

# StandardScaler
print("\nApplying StandardScaler...")
scaler = StandardScaler().fit(X_train[num_cols])
X_train[num_cols] = scaler.transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(f"Before KNNImputation: train {X_train[X_train.isnull().any(axis=1)].shape[0]} rows, test {X_test[X_test.isnull().any(axis=1)].shape[0]} rows with NaN")

# KNNImputer
print("Applying KNNImputer (K=10)...")
imputer = KNNImputer(n_neighbors=10)
X_train[num_cols] = imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = imputer.transform(X_test[num_cols])

print(f"After KNNImputation: train {X_train[X_train.isnull().any(axis=1)].shape[0]} rows, test {X_test[X_test.isnull().any(axis=1)].shape[0]} rows with NaN")

# ✓ FINAL VALIDATION
assert X_train.isnull().sum().sum() == 0, f"Train has {X_train.isnull().sum().sum()} missing values!"
assert X_test.isnull().sum().sum() == 0, f"Test has {X_test.isnull().sum().sum()} missing values!"

# Save LR model data
file_train = os.path.join(OUTPUT_DIR, "train.csv")
file_test = os.path.join(OUTPUT_DIR, "test.csv")
pd.concat([X_train, y_train], axis=1).to_csv(file_train, index=False)
pd.concat([X_test, y_test], axis=1).to_csv(file_test, index=False)

print(f"\n[Logistic Regression] Saved: {file_train}, {file_test}")
print(f"\n✅ All preprocessing completed successfully!")
print(f"   Train shape: {X_train.shape}, Test shape: {X_test.shape}")


X_train: (1852, 47), y_train: (1852,)
X_test:  (464, 47), y_test:  (464,)

categorical_cols: 17
num_cols: 30

EXTERNAL VALIDATION SETUP: TRAIN (MIMIC) → TEST (eICU)
  Train label: AKI=1 63.8%, AKI=0 36.2%
  Test label:  AKI=1 63.8%, AKI=0 36.2%

--- Capping outliers ---
✓ Outlier capping completed

[Tree Model] Saved: AKI_dataset_for_tree_model/train.csv, AKI_dataset_for_tree_model/test.csv

Before categorical imputation:
  Train missing: 10
  Test missing:  4
After categorical imputation:
  Filled 3 cols with -1 (Unknown)
  Filled 14 cols with 0 (Absence)

Applying StandardScaler...
Before KNNImputation: train 1426 rows, test 366 rows with NaN
Applying KNNImputer (K=10)...
After KNNImputation: train 0 rows, test 0 rows with NaN

[Logistic Regression] Saved: AKI_dataset/train.csv, AKI_dataset/test.csv

✅ All preprocessing completed successfully!
   Train shape: (1852, 47), Test shape: (464, 47)
